# Zero Shot object detection with Grounding DINO 

In [ ]:
#!pip install transformers
#!pip install accelerate
#!pip install supervision 

In [ ]:
#Show the image 
from PIL import Image
import matplotlib.pyplot as plt

# Load image
image = Image.open("RoboterLab.png")

# Plot image
plt.figure(figsize=(12, 8))
plt.imshow(image)
plt.axis("off")
plt.show()

In [ ]:
# Code example to run it taken from Huggingface. Minor adjustments made
# https://huggingface.co/IDEA-Research/grounding-dino-base
import requests

import torch
from PIL import Image
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection 

model_id = "IDEA-Research/grounding-dino-base"
device = "cuda" if torch.cuda.is_available() else "cpu"

processor = AutoProcessor.from_pretrained(model_id)
model = AutoModelForZeroShotObjectDetection.from_pretrained(model_id).to(device)

#image_url = "http://images.cocodataset.org/val2017/000000039769.jpg"
#image = Image.open(requests.get(image_url, stream=True).raw)
image = Image.open("RoboterLab.png")
# Check for cats and remote controls
# VERY important: text queries need to be lowercased + end with a dot
text = "a pot. a robot. pliers. a oscilloscope. a cup. a screen. a sticky note. a scetchbook."

inputs = processor(images=image, text=text, return_tensors="pt").to(device)
with torch.no_grad():
    outputs = model(**inputs)

results = processor.post_process_grounded_object_detection(
    outputs,
    input_ids=inputs.input_ids,
    threshold=0.4,
    text_threshold=0.3,
    target_sizes=[image.size[::-1]]
)

In [ ]:
# Retrieve the first image result
result = results[0]
for box, score, labels in zip(result["boxes"], result["scores"], result["text_labels"]):
    box = [round(x, 2) for x in box.tolist()]
    print(f"Detected {labels} with confidence {round(score.item(), 3)} at location {box}")

# Drawing the BBs on the image  

In [ ]:
import supervision as sv
import numpy as np

# Convert PIL image to numpy
image_np = np.array(image.convert("RGB"))

# Get detection outputs
boxes = result["boxes"].cpu().numpy()
scores = result["scores"].cpu().numpy()
labels = result["text_labels"]

# Create supervision detections
detections = sv.Detections(
    xyxy=boxes,
    confidence=scores,
    class_id=np.arange(len(labels))
)

# Create label strings
label_texts = [
    f"{label} {score:.2f}"
    for label, score in zip(labels, scores)
]

# Annotate
box_annotator = sv.BoxAnnotator()
label_annotator = sv.LabelAnnotator()

annotated_image = box_annotator.annotate(
    scene=image_np.copy(),
    detections=detections
)

annotated_image = label_annotator.annotate(
    scene=annotated_image,
    detections=detections,
    labels=label_texts
)

# Show in Jupyter
sv.plot_image(annotated_image, size=(12, 8))